In [2]:
hist="7500000 7500000 7500000 7500000 7500000 7500000 7500000   65620"

import re
import numpy as np

# Replace all whitespaces and new lines with commas
hist=re.sub(r'\s+',',',hist)

# convert the string to a list of integers
hist=list(map(int,hist.split(',')))

sum_hist=sum(hist)*2
print("Sum of histogram:",sum_hist)

remaining=sum_hist-42100000
print("Remaining:",remaining)

remaining_mb=remaining/1000*38
print("Remaining TB:",remaining_mb/1000000)

samples_per_minute=49000
minutes=remaining/samples_per_minute
print("Remaining hours: ",minutes/60)

Sum of histogram: 105131240
Remaining: 63031240
Remaining TB: 2.39518712
Remaining hours:  21.439197278911568


In [3]:
from fretboardnonredundant import get_guitarnote_delays,get_butterworth_group_delay
midi_note=45
delays=[]
delays.append(get_butterworth_group_delay(midi_note,4,6,48000))
delays.append(get_butterworth_group_delay(midi_note,3,6,48000))
delays.append(get_butterworth_group_delay(midi_note,2,6,48000))
delays.append(get_butterworth_group_delay(midi_note,1,6,48000))
print(delays)

[208, 277, 416, 833]


In [ ]:
from scipy import io
from fretboardnonredundant import FretBoard
import numpy
import matplotlib.pyplot as plt
import wave
from common import plot_heatmap,reshape_to_nn_input
sampleRate,input_signal_test=io.wavfile.read('../../assets/testdata/E-G-A-B-chords.wav')
# sampleRate,input_signal_test=io.wavfile.read('../../assets/trainingdata/chords/session_original.wav')
print(sampleRate)
filter=FretBoard(20,sampleRate)
numfilters=filter.get_num_filters()
audio_test=input_signal_test#[:,1];

audiomin=numpy.min(audio_test)
audiomax=numpy.max(audio_test)

audio_test=numpy.multiply((audio_test-audiomin)/(audiomax-audiomin),2)-1
print(len(audio_test))
nFrames =input_signal_test.shape[0]
time=numpy.linspace(0,len(audio_test)/sampleRate,len(audio_test))

plt.plot(time, audio_test, label='Input')

filterbank_out_test=numpy.zeros((numfilters,len(audio_test)))

filter.process(audio_test,filterbank_out_test)

plot_heatmap(filterbank_out_test)

nn_input_test=reshape_to_nn_input(filterbank_out_test)

plot_heatmap(nn_input_test.swapaxes(0,1).max(axis=2),downsample_factor=1)

In [ ]:
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES,OUTPUT_DIM_ONSETS
import numpy as np
from model import build_1d_cnn_model
import tensorflow as tf
from tensorflow import keras
from keras.models import Model
print(nn_input_test.shape)
print(INPUT_SHAPE)




print("test shape",nn_input_test.shape)


# Reshape for the model: (batch_size, time_steps, num_filters, channels)
# For single inference, batch_size = 1, channels = 1
outsample=2035
#outsample=220#370 # E
#outsample=1760 # G
outsample=2255#2145 # A
# outsample=3135#3030 # B

print("Input sample shape",nn_input_test[int(outsample)].shape)
input_for_model = np.expand_dims(nn_input_test[int(outsample)], axis=0)  # Add batch dimension
input_for_model = np.expand_dims(input_for_model, axis=-1) # Add channel dimension


print("Input shape for model ",input_for_model.shape)
print("Building Model")
cnn_model=build_1d_cnn_model(1,INPUT_SHAPE,44)#OUTPUT_DIM_NOTES)
expected_input_shape = input_for_model.shape[1:] 





print("Loading weights")
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch13_valAcc0.9734_valPrec0.8547_valRecall0.7178.weights.h5')
cnn_model.build(input_shape=(None,) + expected_input_shape)

print("Predicting")
predictions_cnn = cnn_model.predict(input_for_model, verbose=0)

# all_predictions_cnn=np.zeros((nn_input_test.shape[0],OUTPUT_DIM_NOTES))
# for i in range(nn_input_test.shape[0]):
#     input_for_model=np.expand_dims(nn_input_test[i],axis=0)
#     input_for_model=np.expand_dims(input_for_model,axis=-1)
#     all_predictions_cnn[i]=cnn_model.predict(input_for_model,verbose=0)


# print("Model Summary")
# cnn_model.summary()
# print("Layer details:")
# # for layer in cnn_model.layers:
# #     print(layer.name, layer.output.shape, layer.input.shape)
# print("Building activation model")
# layer_outputs = [layer.output for layer in cnn_model.layers]

# activation_model=Model(inputs=cnn_model.layers[0].input, outputs=layer_outputs)
# layeractivations=activation_model.predict(input_for_model)



predictions_cnn=np.swapaxes(predictions_cnn,0,1)
print('predictions shape'+str(predictions_cnn.shape))
pred=np.repeat(predictions_cnn,256,axis=1)
print(pred.shape)
#plot_heatmap(pred)

# print(predictions_cnn)
xpred=range(1,len(predictions_cnn))
print(len(predictions_cnn))
print(predictions_cnn.shape)


def midi_to_note(midi_num):
    notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    note_name = notes[midi_num % 12]
    octave = (midi_num // 12) - 1
    return f"{note_name}{octave}"

# Create a wider figure
plt.figure(figsize=(20, 6))

# Plot with MIDI note numbers on x-axis
plt.plot(range(40,84), predictions_cnn[range(0,44)], linewidth=2)
plt.grid(True)
plt.xlabel('MIDI Note Number')
plt.ylabel('Probability')
plt.title('Model Predictions')

# Add x-ticks with both MIDI numbers and note names
x_ticks = range(40, 84, 1)
x_labels = [f"{n}\n{midi_to_note(n)}" for n in x_ticks]
plt.xticks(x_ticks, x_labels, rotation=45)

# Adjust layout to prevent label cutoff
plt.tight_layout()
plt.show()


# # Create a wider figure
# plt.figure(figsize=(15, 5))

# # Plot with MIDI note numbers on x-axis
# plt.plot(range(38,OUTPUT_DIM_NOTES), predictions_cnn[range(38,OUTPUT_DIM_NOTES)], linewidth=2)
# plt.grid(True)
# plt.xlabel('MIDI Note Number')
# plt.ylabel('Probability')
# plt.title('Model Predictions')

# # Add x-ticks every 12 notes (one octave)
# plt.xticks(range(38, OUTPUT_DIM_NOTES, 1))

# # Adjust layout to prevent label cutoff
# plt.tight_layout()
# plt.show()

# plt.plot(nn_output[int(outsample/256)])

In [ ]:
plot_heatmap(all_predictions_cnn.swapaxes(0,1),1)

In [ ]:
from datetime import datetime
import os
log_dir="logs/inference/"+datetime.now().strftime("%Y%m%d-%H%M%S")
file_writer=tf.summary.create_file_writer(log_dir)
with file_writer.as_default():
    for i,(activation,layer) in enumerate(zip(layeractivations,cnn_model.layers)):
        layer_name=layer.name
        tf.summary.histogram(f"{layer_name}/activation_distribution",activation,step=0)
        if 'conv' in layer_name.lower():# or 'pool' in layer_name.lower():
            # 
            print("Writing image for layer ",layer_name,activation.shape)
            image_tensor=activation#np.squeeze(activation,axis=0)
            for i in range(image_tensor.shape[3]):
                image_slice=image_tensor[:,:,:,i:i+1]
                slicename=layer_name+f"_filter_{i}"
                print("Image tensor,",slicename, " shape",image_slice.shape)
                
                tf.summary.image(f"{slicename}/featuremaps",image_slice,step=0,max_outputs=16)
            pass

file_writer.close()

# from model import build_cnn_model
# from common import INPUT_SHAPE, OUTPUT_DIM_NOTES
# import numpy as np
# from tensorboard_viz import visualize_model_architecture, visualize_layer_activations, visualize_predictions

# # Your existing code...
# cnn_model = build_cnn_model(INPUT_SHAPE, OUTPUT_DIM_NOTES)
# cnn_model.load_weights('checkpoints/guitarmidi_epoch13_valAcc0.5885.weights.h5')

# # Prepare your input
# input_for_model = np.expand_dims(nn_input_test[int(outsample)], axis=0)
# input_for_model = np.expand_dims(input_for_model, axis=-1)

# # Add TensorBoard visualization
# import datetime
# log_dir = "logs/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")

# # Visualize architecture
# visualize_model_architecture()

# # Visualize activations
# visualize_layer_activations(cnn_model, input_for_model, log_dir)

# # Get predictions
# predictions_cnn = cnn_model.predict(input_for_model, verbose=0)

# # Visualize predictions
# visualize_predictions(predictions_cnn[0], log_dir=log_dir)

# # Launch TensorBoard
# print("Run: tensorboard --logdir=logs")


In [ ]:
test={0:82,1:87}

print(type(test))
print(test)
print(test.keys())